# Credit Risk EDA

Exploratory Data Analysis of the synthetic credit dataset.

**Key questions:**
1. Is the class imbalance realistic (~15-20% default rate)?
2. Do feature distributions differ between defaults and non-defaults?
3. Are feature correlations sensible?
4. Does default rate increase monotonically from grade A to G?
5. How much missingness exists?

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#475569',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'grid.color': '#334155',
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

ROOT = Path('..').resolve()
PLOTS_DIR = ROOT / 'models' / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / 'data' / 'raw' / 'credit_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (80000, 17)


,fico_score,annual_income,dti_ratio,loan_amount,loan_term,interest_rate,employment_length_years,home_ownership,loan_purpose,credit_history_length_years,num_delinquencies_2yrs,revolving_utilization_pct,loan_grade,verification_status,num_open_accounts,num_derogatory_marks,default
0,625,19800.0,11.98,13700.0,60,11.55,7.0,RENT,credit_card,6,3,67.5,C,Not Verified,9,0,0
1,504,40700.0,22.55,16000.0,36,15.66,5.0,MORTGAGE,vacation,5,4,53.8,D,Source Verified,6,0,0
2,658,101600.0,17.12,8000.0,36,11.97,5.0,RENT,debt_consolidation,11,0,12.4,B,Verified,9,0,0
3,719,42400.0,21.45,9300.0,36,12.62,7.0,RENT,debt_consolidation,8,0,54.4,B,Verified,12,1,0
4,386,42200.0,25.93,23600.0,36,21.89,2.0,RENT,debt_consolidation,1,5,79.2,E,Source Verified,5,0,1


## 1. Class Balance

In [2]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['default'].value_counts()
bars = ax.bar(['Non-Default (0)', 'Default (1)'],
               [counts[0], counts[1]],
               color=['#22c55e', '#ef4444'], alpha=0.85, width=0.5)

for bar, count in zip(bars, [counts[0], counts[1]]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}\n({count/len(df):.1%})',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Class Balance — Default vs Non-Default', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Count', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_class_balance.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Default rate: {df["default"].mean():.1%}')

Default rate: 19.4%


C:\Users\e\AppData\Local\Temp\ipykernel_23060\1646431266.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Distribution Plots by Default Status

In [3]:
features_to_plot = [
    ('fico_score', 'FICO Score', '#60a5fa'),
    ('dti_ratio', 'DTI Ratio (%)', '#f59e0b'),
    ('interest_rate', 'Interest Rate (%)', '#a78bfa'),
    ('revolving_utilization_pct', 'Revolving Utilization (%)', '#34d399'),
    ('annual_income', 'Annual Income ($)', '#fb7185'),
    ('loan_amount', 'Loan Amount ($)', '#22d3ee'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

defaults = df[df['default'] == 1]
non_defaults = df[df['default'] == 0]

for ax, (feat, label, color) in zip(axes, features_to_plot):
    ax.hist(non_defaults[feat].dropna(), bins=50, alpha=0.6,
            color='#22c55e', label='Non-Default', density=True)
    ax.hist(defaults[feat].dropna(), bins=50, alpha=0.6,
            color='#ef4444', label='Default', density=True)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8, facecolor='#1e293b', edgecolor='#475569')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle('Feature Distributions by Default Status', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_distributions.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

C:\Users\e\AppData\Local\Temp\ipykernel_23060\1863305660.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Correlation Heatmap

In [4]:
numeric_cols = [
    'fico_score', 'annual_income', 'dti_ratio', 'loan_amount',
    'loan_term', 'interest_rate', 'employment_length_years',
    'credit_history_length_years', 'num_delinquencies_2yrs',
    'revolving_utilization_pct', 'num_open_accounts',
    'num_derogatory_marks', 'default'
]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))

cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(
    corr, mask=mask, ax=ax, cmap=cmap,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.5, linecolor='#0f172a',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(labelsize=9)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_correlation.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print('\nTop correlations with default:')
print(corr['default'].sort_values().to_string())


Top correlations with default:
fico_score                    -0.452305
credit_history_length_years   -0.277372
num_open_accounts             -0.263920
annual_income                 -0.243703
employment_length_years       -0.211738
loan_term                      0.002037
num_derogatory_marks           0.078420
loan_amount                    0.151478
dti_ratio                      0.342855
revolving_utilization_pct      0.343951
num_delinquencies_2yrs         0.409205
interest_rate                  0.419193
default                        1.000000


C:\Users\e\AppData\Local\Temp\ipykernel_23060\873600454.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Default Rate by Loan Grade (Monotonicity Check)

In [5]:
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
grade_stats = df.groupby('loan_grade').agg(
    default_rate=('default', 'mean'),
    count=('default', 'count')
).reindex(grade_order).dropna()

fig, ax1 = plt.subplots(figsize=(8, 5))

color_map = ['#22c55e', '#84cc16', '#eab308', '#f97316', '#ef4444', '#dc2626', '#7f1d1d']
bars = ax1.bar(grade_stats.index, grade_stats['default_rate'] * 100,
               color=color_map[:len(grade_stats)], alpha=0.85)

for bar, rate in zip(bars, grade_stats['default_rate']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{rate:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.set_xlabel('Loan Grade', fontsize=12)
ax1.set_ylabel('Default Rate (%)', fontsize=12)
ax1.set_title('Default Rate by Loan Grade\n(should be monotonically increasing A → G)', fontsize=13, fontweight='bold')
ax1.spines[['top', 'right']].set_visible(False)
ax1.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_grade_default_rate.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print('Default rate by grade:')
print(grade_stats[['default_rate', 'count']].to_string())

# Check monotonicity
rates = grade_stats['default_rate'].values
is_monotonic = all(rates[i] <= rates[i+1] for i in range(len(rates)-1))
print(f'\n✓ Monotonically increasing: {is_monotonic}')

Default rate by grade:
            default_rate    count
loan_grade                       
A               0.014141  12800.0
B               0.061336  23200.0
C               0.168426  21600.0
D               0.341484  12800.0
E               0.547344   6400.0
F               0.755000   3200.0

✓ Monotonically increasing: True


C:\Users\e\AppData\Local\Temp\ipykernel_23060\3639758620.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Missing Value Analysis

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Percentage', ascending=True)

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(missing_df.index, missing_df['Percentage'], color='#f59e0b', alpha=0.85)
    for bar, pct in zip(bars, missing_df['Percentage']):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=10)
    ax.set_xlabel('Missing (%)', fontsize=12)
    ax.set_title('Missing Values by Feature', fontsize=13, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'eda_missing.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(missing_df.to_string())
else:
    print('No missing values found.')

                           Count  Percentage
revolving_utilization_pct   1600         2.0
employment_length_years     2000         2.5


C:\Users\e\AppData\Local\Temp\ipykernel_23060\4103592026.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary Stats

In [7]:
print('EDA Summary')
print('=' * 50)
print(f'Total rows:     {len(df):,}')
print(f'Default rate:   {df["default"].mean():.1%}')
print(f'Non-defaults:   {(df["default"]==0).sum():,}')
print(f'Defaults:       {(df["default"]==1).sum():,}')
print(f'\nKey correlations with default:')
numeric_cols_no_target = [c for c in numeric_cols if c != 'default']
corr_with_target = df[numeric_cols].corr()['default'].drop('default')
print(corr_with_target.sort_values().to_string())

EDA Summary
Total rows:     80,000
Default rate:   19.4%
Non-defaults:   64,468
Defaults:       15,532

Key correlations with default:


fico_score                    -0.452305
credit_history_length_years   -0.277372
num_open_accounts             -0.263920
annual_income                 -0.243703
employment_length_years       -0.211738
loan_term                      0.002037
num_derogatory_marks           0.078420
loan_amount                    0.151478
dti_ratio                      0.342855
revolving_utilization_pct      0.343951
num_delinquencies_2yrs         0.409205
interest_rate                  0.419193
